# 类与继承

学习目标：能用类封装独立状态与共享行为，掌握私有元素、静态初始化及继承初始化顺序。

前置知识：对象、this、构造调用、原型链、访问器与函数参数。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 文件使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。以下命令均从此目录运行；每个入口使用独立 Node.js 进程。

配套脚本：位于 scripts/14-classes/。

1. [instances.mjs](scripts/14-classes/instances.mjs)：独立字段与共享方法。
2. [without-new-error.mjs](scripts/14-classes/without-new-error.mjs)：类不能普通调用的独立反例。
3. [static-members.mjs](scripts/14-classes/static-members.mjs)：静态字段、方法与初始化顺序。
4. [private-state.mjs](scripts/14-classes/private-state.mjs)：私有字段、方法与公开访问器。
5. [private-receiver-error.mjs](scripts/14-classes/private-receiver-error.mjs)：访问私有字段的方法对接收者的限制。
6. [inheritance.mjs](scripts/14-classes/inheritance.mjs)：父类初始化、覆盖与静态继承。
7. [before-super-error.mjs](scripts/14-classes/before-super-error.mjs)：super 前访问 this 的独立反例。
8. [initialization.mjs](scripts/14-classes/initialization.mjs)：字段顺序和 setter 边界。
9. [composition.mjs](scripts/14-classes/composition.mjs)：通过组合替换格式化能力。

## 1 类、实例字段和方法

class 把构造过程、字段和方法组织在一起，类声明在初始化之前不能访问，类体自动采用严格模式。必须使用 new 构造类，不能把类当普通函数调用。

constructor 初始化本次实例，公开实例字段也属于每个实例；实例方法保存在类的 prototype 上，供实例共享。字段初始化表达式会在每次创建实例时求值，因此字段中的新数组不会在实例之间共享。类建立在原型机制上，同时增加私有元素、严格调用条件等语义，不能简单等同于任意函数替换。

[instances.mjs](scripts/14-classes/instances.mjs)：

```javascript
class Course {
  notes = [];
  constructor(title) { this.title = title; }
  add(note) { this.notes.push(note); }
  describe() { return this.title + ":" + this.notes.length; }
}
const js = new Course("JS");
const ts = new Course("TS");
js.add("类");
console.log(js.describe(), ts.describe());
console.log(js.add === ts.add, js.notes === ts.notes);
console.log(Object.hasOwn(js, "notes"), Object.hasOwn(js, "add"));
console.log(Object.getPrototypeOf(js) === Course.prototype);

// 按本例输入运行，输出依次为：
// JS:1 TS:0
// true false
// true false
// true
```

Step 1：运行本节示例。

```bash
node scripts/14-classes/instances.mjs
```

[without-new-error.mjs](scripts/14-classes/without-new-error.mjs)：

```javascript
class Course {}
Course();

// 独立运行：退出状态为 1；诊断包含 TypeError；cannot be invoked without 'new'。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/14-classes/without-new-error.mjs
```

## 2 静态成员和静态初始化块

static 声明的字段或方法属于类对象，不属于实例。静态方法中的 this 通常取调用该方法的类对象，仍然遵循调用形式规则。static 初始化块用于在类求值时执行多条初始化语句，可访问类的私有静态状态。

静态字段初始化和静态块按源码顺序执行一次，不会在每个实例创建时重复。静态块有自身作用域，不是异步函数，不能把 await 当作可直接使用的初始化语法。

[static-members.mjs](scripts/14-classes/static-members.mjs)：

```javascript
const events = [];
class Catalog {
  static prefix = (events.push("字段"), "JS");
  static titles;
  static {
    events.push("静态块");
    this.titles = [this.prefix + " 基础"];
  }
  static count() { return this.titles.length; }
  constructor() { events.push("实例"); }
}
const first = new Catalog();
new Catalog();
console.log(events.join(","));
console.log(Catalog.count(), Catalog.titles[0], typeof first.count);

// 按本例输入运行，输出依次为：
// 字段,静态块,实例,实例
// 1 JS 基础 undefined
```

Step 1：运行本节示例。

```bash
node scripts/14-classes/static-members.mjs
```

## 3 私有元素与访问器

以 # 开头的私有名字必须在类体声明；私有字段、方法、访问器都不能通过字符串键访问。公开 getter/setter 可以提供受控的属性接口，内部再读写私有状态。下面 amount 表示本例给定的合法计数值；公开 setter 调用私有方法 #save 保存它，getter 读取同一私有字段。#save 在这里用于展示私有方法的声明和类内调用。

私有元素不是普通属性，不在 Object.keys 中；类内可以访问同一个类创建的其他对象的私有状态，但接收者必须确实拥有对应私有元素。单纯借用方法到空对象上会抛 TypeError。子类不能直接使用父类的私有名字，应调用父类公开或可继承的方法；同拼写的子类私有名字也是另一个名字。

[private-state.mjs](scripts/14-classes/private-state.mjs)：

```javascript
class Progress {
  #amount = 0;
  #save(value) { this.#amount = value; }
  get amount() { return this.#amount; }
  set amount(value) { this.#save(value); }
  sameAs(other) { return this.#amount === other.#amount; }
}
const first = new Progress();
const second = new Progress();
first.amount = 3;
second.amount = 3;
console.log(first.amount, first.sameAs(second));
console.log(Object.keys(first).length, first["#amount"]);

// 按本例输入运行，输出依次为：
// 3 true
// 0 undefined
```

Step 1：运行本节示例。

```bash
node scripts/14-classes/private-state.mjs
```

[private-receiver-error.mjs](scripts/14-classes/private-receiver-error.mjs)：

```javascript
class Vault {
  #value = 7;
  read() { return this.#value; }
}
Vault.prototype.read.call({});

// 独立运行：退出状态为 1；诊断包含 TypeError；Cannot read private member #value。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/14-classes/private-receiver-error.mjs
```

## 4 extends、super 与方法覆盖

Lab.kind 与 lab.describe() 走的不是同一条链。下面把构造函数对象与实例方法所在的原型对象分开。

extends 同时连接两条关系：子类 prototype 继承父类 prototype，子类构造函数对象继承父类构造函数对象，因此公开静态成员也能被继承。子类若显式编写 constructor，在常规创建实例的写法中，访问 this 前必须先执行 super()，让父类构造过程初始化实例。

super.method() 从父类原型寻找方法，调用时接收者仍是当前实例。重写同名方法可以先调用父类实现，再加入子类行为。父类方法若访问自己的私有字段，在经过父类构造的子类实例上仍可正常工作。

![extends 同时连接两条继承关系。纵向虚线是 prototype 属性；横向实线是对象的原型关系。](image/illustration/14-01-class-inheritance-links.svg)

图示说明：依据类定义的继承连接规则自绘；图只展示本例的继承连接；实例字段和私有元素的初始化仍由构造过程完成。

对照下面两次 getPrototypeOf 检查，再观察 super.describe() 如何与当前 lab 的 minutes 组合。

[inheritance.mjs](scripts/14-classes/inheritance.mjs)：

```javascript
class Lesson {
  static kind = "教学";
  #title;
  constructor(title) { this.#title = title; }
  describe() { return this.#title; }
}
class Lab extends Lesson {
  constructor(title, minutes) {
    super(title);
    this.minutes = minutes;
  }
  describe() { return super.describe() + ":" + this.minutes; }
}
const lab = new Lab("类", 15);
console.log(lab.describe(), lab instanceof Lab, lab instanceof Lesson);
console.log(Lab.kind, Object.getPrototypeOf(Lab) === Lesson);
console.log(Object.getPrototypeOf(Lab.prototype) === Lesson.prototype);

// 按本例输入运行，输出依次为：
// 类:15 true true
// 教学 true
// true
```

Step 1：运行本节示例。

```bash
node scripts/14-classes/inheritance.mjs
```

[before-super-error.mjs](scripts/14-classes/before-super-error.mjs)：

```javascript
class Base {}
class Child extends Base {
  constructor() {
    this.ready = true;
    super();
  }
}
new Child();

// 独立运行：退出状态为 1；诊断包含 ReferenceError；Must call super constructor。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/14-classes/before-super-error.mjs
```

## 5 初始化顺序与覆盖的边界

对下面的普通基类和派生类：先进入子类构造体执行 super 前的语句；super 开始创建并初始化实例，先运行基类字段、再运行基类构造体；super 返回后、下一条子类构造语句之前初始化子类字段，然后继续子类构造体。

因此父类构造中不宜调用依赖子类字段的可覆盖方法：该方法可以已经被覆盖，但子类字段尚未初始化。公开字段通过定义自有属性初始化，不等于执行一次普通赋值；如果父类原型有同名 setter，子类字段定义不会调用它。

[initialization.mjs](scripts/14-classes/initialization.mjs)：

```javascript
const events = [];
class Base {
  base = (events.push("基类字段"), 1);
  // 有意在基类构造中调用可覆盖方法，观察子类字段此时尚未就绪。
  constructor() { events.push("基类构造:" + this.describe()); }
  describe() { return "base"; }
  set count(value) { events.push("setter:" + value); }
}
class Child extends Base {
  detail = (events.push("子类字段"), "ready");
  // 字段定义创建自有属性，不调用基类原型上同名的 setter。
  count = 2;
  constructor() {
    events.push("super前");
    // super 完成基类初始化后，再初始化子类字段，才继续下一行。
    super();
    events.push("super后:" + this.detail);
  }
  describe() { return this.detail; }
}
const child = new Child();
console.log(events.join(","));
console.log(child.count, Object.hasOwn(child, "count"));

// 按本例输入运行，输出依次为：
// super前,基类字段,基类构造:undefined,子类字段,super后:ready
// 2 true
```

Step 1：运行本节示例。

```bash
node scripts/14-classes/initialization.mjs
```

## 6 用组合替换不必要的继承

如果两个对象只是需要不同的格式化方式，而没有共同的类型关系，可以把格式化函数交给对象使用。下例 Reporter 保存 formatter 并调用它，这是组合：能力可以替换，而无须创建许多格式化子类。

继承适合确实需要共同接口并能遵守父类约定的情况；组合适合独立变化的能力。这是本例设计判断，语言本身不会替你证明子类满足父类的业务约束。

[composition.mjs](scripts/14-classes/composition.mjs)：

```javascript
class Reporter {
  constructor(formatter) { this.formatter = formatter; }
  render(title) { return this.formatter(title); }
}
const plain = new Reporter(title => title);
const labeled = new Reporter(title => "课程:" + title);
console.log(plain.render("继承"), labeled.render("组合"));

// 按本例输入运行，输出依次为：
// 继承 课程:组合
```

Step 1：运行本节示例。

```bash
node scripts/14-classes/composition.mjs
```

## 本章小结

- 实例字段保存各自状态，实例方法通过 prototype 共享；static 属于类对象。
- 私有名字与普通属性不同，方法借用不能绕过接收者检查。
- super、字段初始化与构造体有明确先后顺序，组合可以独立替换能力。

## 练习

1. 为 Course 添加公开 getter 返回 notes.length。可核对标准：给一个实例增加两条笔记后 getter 为 2，另一个实例仍为 0。
2. 为 Progress 增加只读 getter doubled，返回私有计数的两倍；保留原 amount 访问器。分别给两个实例写入 3 与 5，可核对标准：amount 仍为 3、5，doubled 为 6、10。
3. 在 initialization.mjs 中让父类构造不再调用可覆盖的 describe，在构造完成后调用。可核对标准：不再读到未初始化的 detail，最终 describe 返回 ready。

### 提示

1. 新建两个 Course，各自 notes 初始为空。
2. 在类体中读取 #amount，不用字符串键。
3. 只改变父类构造体中调用 describe 的位置，保留字段初始化顺序。


### 参考解析

1. 在 Course 中增加 `get count() { return this.notes.length; }`；向第一个实例 add 两次后，两个实例 count 分别为 2 和 0。
2. `get doubled() { return this.#amount * 2; }` 只计算读取结果，不给 #amount 赋值，所以 amount 保留 3 和 5，而 doubled 分别为 6 和 10。
3. 基类构造体只记录“基类构造”；在 `const child = new Child()` 后调用 child.describe()，此时返回 ready。其余事件顺序仍是 super前、基类字段、基类构造、子类字段、super后:ready。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TC39 官方 ECMAScript 2025 分页版 | [§15.7 类字段、方法、静态块和 ClassDefinitionEvaluation](https://tc39.es/ecma262/2025/multipage/ecmascript-language-functions-and-classes.html#sec-class-definitions)；[§7.3.33 实例元素初始化](https://tc39.es/ecma262/2025/multipage/abstract-operations.html#sec-initializeinstanceelements)；[§7.3.30 私有元素访问](https://tc39.es/ecma262/2025/multipage/abstract-operations.html#sec-privateget)；[§10.2.2 构造与字段顺序](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-ecmascript-function-objects-construct-argumentslist-newtarget)。 |
| MDN 用法对照 | [实例方法、私有字段、静态属性与 extends](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Using_classes)。 |
